# LLM API 与 Ollama - 超越 OpenAI

_重要：如果你对 API 本身还不太熟悉，以及对 PC 或 Mac 上的环境变量还不熟，请先复习指南 4「Technical Foundations」中的 API 部分，再继续本指南（指南 4 的主题 3 和 5）。_

## 使用 OpenAI 以外模型的关键背景——请先阅读！

贯穿整门课，我们使用 API 连接地球上最强的 LLM。

这些 LLM 背后的公司，如 OpenAI、Anthropic、Google 和 DeepSeek，构建了 Web 端点。你通过向某个 Web 地址发起 HTTP 请求，并传入关于提示的所有信息，来调用它们的模型。

但如果每次想调用 API 都要自己构建 HTTP 请求，会很痛苦。

为了简化，OpenAI 团队写了一个 Python 工具，称为「Python Client Library」，它封装了 HTTP 调用。所以你写 Python 代码，它去调用 Web。

而这，就是库 `openai`。

### 什么是 `openai` Python 客户端库

它是：
- 一个轻量的 Python 工具
- 把你的 Python 请求变成 HTTP 调用
- 把 HTTP 调用返回的结果转换成 Python 对象

### 它不是什么

- 它并不包含实际运行大型语言模型的代码！没有 GPT 代码！它只是发起 Web 请求
- 没有科学计算代码，也没有特别针对 OpenAI 的专用逻辑

### 如何使用：

```python
# Create an OpenAI python client for making web calls to OpenAI
openai = OpenAI()

# Make the call
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=[{"role":"user", "content": "what is 2+2?"}])

# Print the result
print(response.choices[0].message.content)
```

### 这做了什么

当你发起 Python 调用：`openai.chat.completions.create()`  
它只是向这个 URL 发起 Web 请求：`https://api.openai.com/v1/chat/completions`  
并把响应转换成 Python 对象。

就是这样。

如果你直接发起 [Web HTTP 调用](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=curl)，这是 API 文档  
如果你使用 [Python Client Library](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=python)，这是同一份 API 文档

## 有了这些背景——我如何使用其他 LLM？

事实证明——超级简单！

所有其他主要 LLM 都有与 OpenAI 兼容的 API 端点。

于是 OpenAI 帮了大家一个忙：他们说，嘿——你们都可以用我们的工具把 Python 转成 Web 请求。我们允许你把工具从调用 `https://api.openai/com/v1` 改成调用你指定的任何 Web 地址。

所以你甚至可以用 OpenAI 工具调用并非 OpenAI 的模型，像这样：

`not_actually_openai = OpenAI(base_url="https://somewhere.completely.different/", api_key="another_providers_key")`

重要的是要理解：这段 OpenAI 代码只是一个向端点发起 HTTP 调用的工具。所以即使我们用的是 OpenAI 团队的代码，也可以用它调用 OpenAI 以外的模型。

以下是各大提供商的所有 OpenAI 兼容端点。甚至包括在本地使用 Ollama。Ollama 在你的本地机器上提供端点，并且也做成了 OpenAI 兼容——非常方便。

```python
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
```

## 这里是 Gemini、DeepSeek、Ollama 和 OpenRouter 的示例

### 示例 1：用 Gemini 代替 OpenAI

1. 访问 Google Studio 开设账户：https://aistudio.google.com/  
2. 把你的密钥作为 GOOGLE_API_KEY 添加到 `.env`  
3. 再作为 GEMINI_API_KEY 添加第二次到 `.env`——之后会有用。

然后：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 示例 2：用 DeepSeek API 代替 OpenAI（便宜，且只需预付 $2）

1. 访问 DeepSeek API 开设账户：https://platform.deepseek.com/  
2. 你需要先充值最低 $2。  
3. 把你的密钥作为 DEEPSEEK_API_KEY 添加到 `.env`  

然后：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
response = deepseek.chat.completions.create(model="deepseek-chat", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 示例 3：用 Ollama 免费且本地地代替 OpenAI

Ollama 允许你在本地运行模型；它在你的机器上提供 OpenAI 兼容的 API。  
Ollama 没有 API key；没有第三方拿着你的信用卡，所以不需要任何密钥。

1. 如果你是 Ollama 新手，按这里的说明安装：https://ollama.com   
2. 然后在 Cursor Terminal 中执行 `ollama run llama3.2` 来与 Llama 3.2 聊天  
注意：不要使用 llama3.3 或 llama4——这些是巨型模型，不是为家用计算设计的！它们会占满你的磁盘。  

然后：

```python
!ollama pull llama3.2

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 示例 4：使用流行服务 [OpenRouter](https://openrouter.ai)，其计费流程比 OpenAI 更简单

OpenRouter 非常方便：它让你免费访问许多模型，并以便捷的小额预付访问付费模型。

1. 在 https://openrouter.ai 注册
2. 按需添加最低预付余额
3. 把你的密钥作为 OPENROUTER_API_KEY 添加到 `.env` 文件

然后：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
response = openrouter.chat.completions.create(model="openai/gpt-4.1-nano", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```


### 在 Agent Frameworks 中使用不同的 API 提供商

Agent Frameworks 让在这些提供商之间切换变得容易。你可以在课程的任何时候切换 LLM 并选择不同的模型。下面有关于它们各自的更多说明。对于 OpenAI Agents SDK，见本 notebook 后面的一节。对于 CrewAI，我们在课程中会讲，但很简单：只需使用 LiteLLM 期望的模型完整路径。

## API 的费用

每次 API 调用的费用确实非常低——本课程中我们使用的模型，多数调用只需几分之一美分。

但极其重要的是要注意：

1. 一个复杂的 Agentic 项目可能涉及许多次 LLM 调用——也许 20–30 次——所以费用会累积。设置限额并监控用量很重要。

2. 使用 Agentic AI 时，存在 Agent 陷入循环或执行超出预期处理的风险。你应该监控 API 用量，并且永远不要投入超过你能接受的预算。有些 API 有「自动充值」设置，会自动从你的卡扣款——我强烈建议关掉它。

3. 你应该只花自己能接受的钱。如果你愿意，可以用 Ollama 作为免费替代。DeepSeek、Gemini 2.5 Flash 和 gpt-4.1-nano 明显更便宜。

请记住：这些 LLM 调用通常涉及数万亿次浮点运算——总得有人为电费买单！

### Ollama：付费 API 的免费替代（但请看关于 llama 版本的警告）

Ollama 是在你机器上本地运行的产品。它可以运行开源模型，并在你的电脑上提供与 OpenAI 兼容的 API 端点。

首先，访问以下地址下载 Ollama：
https://ollama.com

然后在 Cursor 的 Terminal 中（View 菜单 >> Terminal），运行此命令下载模型：

```shell
ollama pull llama3.2
```

警告：小心不要使用 llama3.3 或 llama4——这些模型大得多，不适合家用电脑。

现在，任何时候我们有像这样的代码：  
`openai = OpenAI()`  
你都可以用这个作为直接替换：  
`openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
并且把像 **gpt-4o-mini** 这样的模型名替换为 **llama3.2**。  

你不需要为此在 .env 文件中放任何东西；使用 Ollama 时，一切都在你的电脑上运行。你没有调用云上的第三方，没人拿着你的信用卡信息，所以不需要密钥！上面的代码 `api_key='ollama'` 之所以需要，只是因为 OpenAI 客户端库期望传入 api_key，但该值会被 Ollama 忽略。

下面是完整示例：

```python
# You need to do this one time on your computer
!ollama pull llama3.2

from openai import OpenAI
MODEL = "llama3.2"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

你需要做类似的改动才能在任意 Agent Framework 中使用 Ollama——你应该能搜到确切示例，或问我。

### OpenRouter：面向 OpenAI 及其他模型的便捷网关平台

OpenRouter 是第三方服务，允许你连接包括 OpenAI 在内的广泛 LLM。

它以更简单的计费流程著称，对一些美国以外的国家可能更方便。

首先，看看他们的网站：  
https://openrouter.ai/

然后，看看他们的快速入门：  
https://openrouter.ai/docs/quickstart

并把你的密钥添加到 .env 文件：  
```shell
OPENROUTER_API_KEY=sk-or....
```

现在，任何时候你有像这样的代码：  
```python
MODEL = "gpt-4o-mini"
openai = OpenAI()
```

你可以用像这样的代码替换它：

```python
MODEL = "openai/gpt-4o-mini"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

你需要做类似的改动才能在任意 Agent Framework 中使用 OpenRouter——你应该能搜到确切示例，或问我。

## OpenAI Agents SDK - 具体说明

使用 OpenAI Agents SDK（第 2 周和第 6 周）时，使用 OpenAI 自己提供的任何模型特别容易。只需传入模型名：

`agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")`

你也可以用有 OpenAI 兼容 API 的任何其他提供商替换。按这 3 步做：

```python
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
```

然后在创建 Agent 时提供这个模型即可。

`agent = Agent(name="Jokester", instructions="You are a joke teller", model=deepseek_model)`

对任何其他 OpenAI 兼容 API，你可以用类似方法，同样 3 步：

```python
# extra imports
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

# Step 1: specify the base URL endpoints where the provider offers an OpenAI compatible API
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Step 2: Create an AsyncOpenAI object for that endpoint
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Step 3: Create a model object to provide when creating an Agent
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
grok_3_model = OpenAIChatCompletionsModel(model="grok-3-mini-beta", openai_client=openrouter_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
grok_3_via_openrouter_model = OpenAIChatCompletionsModel(model="x-ai/grok-3-mini-beta", openai_client=openrouter_client)
llama_3_2_local_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
```

### 在 OpenAI Agents SDK 中使用 Azure

见此处说明：  
https://techcommunity.microsoft.com/blog/azure-ai-services-blog/use-azure-openai-and-apim-with-the-openai-agents-sdk/4392537

例如这样：
```python
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client
from dotenv import load_dotenv
import os
 
# Load environment variables
load_dotenv()
 
# Create OpenAI client using Azure OpenAI
openai_client = AsyncAzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT")
)
 
# Set the default OpenAI client for the Agents SDK
set_default_openai_client(openai_client)
```

## CrewAI 设置

这里是 Crew 关于 LLM 连接的文档，含所有模型要用的模型名。正如学员 Sadan S. 指出的（谢谢！），值得知道：对 Google 你需要使用环境变量 `GEMINI_API_KEY`，而不是 `GOOGLE_API_KEY`：

https://docs.crewai.com/concepts/llms

这里是他们带有更多信息的教程：

https://docs.crewai.com/how-to/llm-connections

## LangGraph 设置

要在 LangGraph 中使用 Ollama（其他模型也类似）：  
https://python.langchain.com/docs/integrations/chat/ollama/#installation

首先添加包：  
`uv add langchain-ollama`

然后在 lab 中做这个替换：   
```python
from langchain_ollama import ChatOllama
# llm = ChatOpenAI(model="gpt-4o-mini")
llm = ChatOllama(model="gemma3:4b")
```

显然要事先运行 `!ollama pull gemma3:4b`（或你选用的模型）。

非常感谢 Miroslav P. 补充了这些，也感谢 Arvin F. 提出问题！

## 在 LangGraph 中使用其他模型

只需遵循与上面相同的做法，但使用这里的任意模型：  
https://python.langchain.com/docs/integrations/chat/



## 在 AutoGen 中使用其他模型

这里是 Miroslav P. 的又一份贡献（谢谢！），关于在 AutoGen 中使用 Ollama + 本地模型；Miroslav 有一个很棒的示例，展示 gemma3 表现良好。

```python
# model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
 
from autogen_ext.models.ollama import OllamaChatCompletionClient
 
model_client = OllamaChatCompletionClient(
    model="gemma3:4b",
    model_info={
        "vision": True,
        "function_calling": False,
        "json_output": True,
        "family": "unknown",
    },
)
```

## 值得记住

1. 如果你希望用 Ollama 在本地运行模型，你可能会发现较小模型在更高级的项目上会吃力。你需要尝试不同的模型大小和能力，并且可能需要很多耐心才能找到表现良好的组合。我预计我们的几个项目对 llama3.2 来说太有挑战。作为替代，可以考虑 openrouter.ai 上的免费模型，或几乎免费的非常便宜的模型——比如 DeepSeek。

2. Chat 模型往往比 Reasoning 模型表现更好，因为 Reasoning 模型可能对某些任务「想太多」。实验很重要。更大并不总是更好……

3. 这容易混淆，但有两个听起来相似的不同提供商！  
- Grok 是 Elon Musk 的 X 的 LLM
- Groq 是用于开源模型快速推理的平台

有学员向我指出：「Groq」出现得更早！
